# VAANI — Colab Runner

Before running anything: **Runtime → Change runtime type → T4 GPU → Save.**

Run the cells top to bottom. Step 2 clones the code automatically (URL is
already filled in) — only step 4 needs you to upload something: your
recorded reference voice clip.

## 1. Confirm you actually got a GPU

In [ ]:
!nvidia-smi

## 2. Get the code onto this machine

The repo URL is already filled in below — just run this cell. (If it's a
private repo, this clone will fail with a 404/permission error; in that
case skip to the fallback cell after this one and upload a zip instead.)

In [ ]:
REPO_URL = "https://github.com/YASHASWINIKSHRESTHA/Vaani_AI.git"
import os
if not os.path.isdir("vaani"):
    !git clone {REPO_URL} vaani
%cd vaani
!ls

**Fallback only** — run this instead of the cell above if the clone failed
(e.g. private repo without credentials configured in this Colab session).

In [ ]:
# Fallback: upload a zip of your local vaani/ folder
# (on your machine: right-click the vaani folder -> Compress/Send to zip -> vaani.zip)
import os
if not os.path.isdir("vaani"):
    from google.colab import files
    uploaded = files.upload()  # select vaani.zip
    import zipfile
    for fname in uploaded.keys():
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall("vaani")
%cd vaani
!ls

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
# If this cell errors with a version-resolver conflict, that's documented
# in GAPS.md section 1b item 6 -- come back and ask for the per-model venv
# fallback instead of trying to force it through.

## 4. Upload your reference voice clip

~15–20 seconds, quiet room, mono, recorded on your phone. Any common audio
format works here (wav/m4a/mp3).

In [ ]:
from google.colab import files
import shutil, os

os.makedirs("samples/reference", exist_ok=True)
uploaded = files.upload()  # select your reference audio file
for fname in uploaded.keys():
    shutil.move(fname, f"samples/reference/{fname}")
print("Saved:", os.listdir("samples/reference"))

## 5. Run the eval

Set `REFERENCE_WAV` to the filename printed above, then run. This synthesizes
every test sentence in all 3 languages, gates each clip, and writes logs/CSVs/wavs.

In [ ]:
REFERENCE_WAV = "samples/reference/CHANGE_ME.wav"  # <-- update to your actual filename
!python -m eval.run_eval --reference-wav "{REFERENCE_WAV}"

## 6. Build the results table

In [ ]:
!python -m eval.build_report

## 7. Generate the MOS listener survey

This samples real generated clips into a CSV. Download it (and the clips
themselves, step 8) and send them to 2–3 people to rate 1–5.

In [ ]:
!python -m eval.generate_mos_survey

## 8. Download everything back to your machine

Bring these files back to Claude to finish the README table, FAILURES.md,
and the summary paragraph from real numbers.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("vaani_logs", "zip", ".", "logs")
shutil.make_archive("vaani_generated_samples", "zip", ".", "samples/generated")

files.download("vaani_logs.zip")
files.download("vaani_generated_samples.zip")
import glob
for f in glob.glob("eval/results_*.csv"):
    files.download(f)
files.download("eval/mos_survey_template.csv")